Data set:
https://www.cancerimagingarchive.net/collection/aml-cytomorphology_mll_helmholtz/


# Downloading the zip file from Google Drive

In [ ]:
!pip install gdown -q
!gdown 1NBeegdfQRGYW82uFD53O9oD4xTNOaHmg
!unzip -q 'PKG - AML-Cytomorphology_MLL_Helmholtz_v1.zip'

Downloading...
From (original): https://drive.google.com/uc?id=1NBeegdfQRGYW82uFD53O9oD4xTNOaHmg
From (redirected): https://drive.google.com/uc?id=1NBeegdfQRGYW82uFD53O9oD4xTNOaHmg&confirm=t&uuid=03bf4d49-083e-4935-b0ce-a98671441d7e
To: /content/PKG - AML-Cytomorphology_MLL_Helmholtz_v1.zip
100% 3.42G/3.42G [00:36<00:00, 95.0MB/s]


# Peparing the dirs and converting images

In [ ]:
import os
import shutil
import random
from tqdm import tqdm
from PIL import Image
import math

def split_dataset(source_dir, train_dir, test_dir, train_ratio=0.7):
    os.makedirs(train_dir, exist_ok=True)
    os.makedirs(test_dir, exist_ok=True)

    for class_folder in tqdm(os.listdir(source_dir), desc="Processing classes"):
        class_path = os.path.join(source_dir, class_folder)
        if not os.path.isdir(class_path):
            continue

        train_class_dir = os.path.join(train_dir, class_folder)
        test_class_dir = os.path.join(test_dir, class_folder)
        os.makedirs(train_class_dir, exist_ok=True)
        os.makedirs(test_class_dir, exist_ok=True)

        patient_folders = [f for f in os.listdir(class_path) if os.path.isdir(os.path.join(class_path, f))]
        num_train_patients = math.ceil(len(patient_folders) * train_ratio)

        random.shuffle(patient_folders)
        train_patients = patient_folders[:num_train_patients]
        test_patients = patient_folders[num_train_patients:]

        for patient in train_patients:
            patient_path = os.path.join(class_path, patient)
            process_patient_folder(patient_path, train_class_dir, patient, "train")

        for patient in tqdm(test_patients, desc=f"Processing test patients for {class_folder}", leave=False):
            patient_path = os.path.join(class_path, patient)
            process_patient_folder(patient_path, test_class_dir, patient, "test")

def process_patient_folder(patient_path, output_dir, patient_id, split_type):
    for img_file in os.listdir(patient_path):
        if img_file.lower().endswith('.tif'):
            src_path = os.path.join(patient_path, img_file)
            new_filename = f"{split_type}_{patient_id}_{img_file[:-4]}.jpg"
            dst_path = os.path.join(output_dir, new_filename)

            try:
                with Image.open(src_path) as img:
                    img = img.convert('RGB')
                    img.save(dst_path, "JPEG")
            except Exception as e:
                print(f"Error processing {src_path}: {e}")

if __name__ == "__main__":
    source_dir = "PKG - AML-Cytomorphology_MLL_Helmholtz_v1/data"
    train_dir = "dataset/train"
    test_dir = "dataset/test"

    split_dataset(source_dir, train_dir, test_dir)
    print("\nDataset split and image conversion complete.")

Processing classes: 100%|██████████| 5/5 [02:27<00:00, 29.44s/it]


Dataset split and image conversion complete.


In [ ]:
!zip -rq dataset.zip dataset -9

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!cp dataset.zip /content/drive/MyDrive/Leukemia/